All necessary inclusions. First, run this cell.

In [1]:
#pragma cling add_include_path("../../lib/")
#include <walimpl.hpp>
#include <syncmem.hpp>
#include <thread>
#include <atomic>

Development of class ***base_bag_of_tasks*** for managing element-by-element squaring of elements of a numerical series.

In [2]:
class base_bag_of_tasks{
public:
    void resize(unsigned size){
        N.resize(size); NxN.resize(size);
        unprocessed.clear(); _ready_to_get = false;
    }
    void add(unsigned id,int n){
        N[id]=n; unprocessed.insert(id);
        if(unprocessed.size()==N.size()) _ready_to_get = true;
    }
    bool ready_to_get(){return _ready_to_get;}
    bool get(unsigned& id, int& n){
        if(unprocessed.empty())return false;
        id = get_rand_unprocessed();  
        n = N[id];
        return true;
    }
    void put(unsigned id,int nxn){
        unprocessed.erase(id);
        NxN[id]=nxn;
    }
public:
    std::vector<int> N;
    std::vector<int> NxN;
private:
    std::set<unsigned> unprocessed;
    bool _ready_to_get;
    unsigned get_rand_unprocessed(){
        int selected = rand() % unprocessed.size();
		auto it = unprocessed.begin(); 
        for (int i = 0; i != selected; i++, it++) {}
        return *it;
    }
};

Testing the ***base_bag_of_tasks*** class.

In [3]:
{
    const int SIZE = 10;
    base_bag_of_tasks tbag;
    
    tbag.resize(SIZE);
    for(int i=0;i<SIZE;i++) tbag.add(i,i);
    
    while(!tbag.ready_to_get())/*wait*/;
    
    unsigned id; int N, NxN;
    while(tbag.get(id,N)){
        NxN = N*N; tbag.put(id,NxN);
    }
    
    for(int i=0;i<SIZE;i++)
        std::cout << tbag.N[i] <<"^2 = " << tbag.NxN[i] << std::endl;
}

0^2 = 0
1^2 = 1
2^2 = 4
3^2 = 9
4^2 = 16
5^2 = 25
6^2 = 36
7^2 = 49
8^2 = 64
9^2 = 81


We refactor class ***base_bag_of_tasks*** into a singleton class ***bag_of_tasks*** with globally synchronized state.

In [4]:
class bag_of_tasks:public templet::state{
public:
    bag_of_tasks(templet::write_ahead_log&l):state(l) { init(); }
    void resize(unsigned size){
        update(_resize, [&](std::ostream&out) {
            out << size;
		},
		[this](std::istream&in) {
            unsigned size; in >> size;
            N.resize(size); NxN.resize(size);
            unprocessed.clear(); _ready_to_get = false;
		});   
    }
    void add(unsigned id,int n){
        update(_add, [&](std::ostream&out) {
            out << id << " " << n;
		},
		[this](std::istream&in) {
            unsigned id; int n; in >> id >> n;
            N[id]=n; unprocessed.insert(id);
            if(unprocessed.size()==N.size()) _ready_to_get = true;
		});     
    }
    bool ready_to_get(){
        update();
        return _ready_to_get;
    }
    bool get(unsigned& id, int& n){
        update();
        if(unprocessed.empty())return false;
        id = get_rand_unprocessed(); n = N[id];
        return true;
    }
    void put(unsigned id,int nxn){
        update(_put, [&](std::ostream&out) {
            out << id << " " << nxn;
		},
		[this](std::istream&in) {
            unsigned id; int nxn; in >> id >> nxn;
            unprocessed.erase(id);
            NxN[id]=nxn;
		});  
    }
public:
    std::vector<int> N;
    std::vector<int> NxN;
private:
    std::set<unsigned> unprocessed;
    bool _ready_to_get;
    unsigned get_rand_unprocessed(){
        int selected = rand() % unprocessed.size();
		auto it = unprocessed.begin(); 
        for (int i = 0; i != selected; i++, it++) {}
        return *it;
    }
private:
	enum {_resize,_add,_put};
	void on_init() override {
		resize(0); add(0,0); put(0,0);
	}
};

We test the operation of class ***bag_of_tasks*** by simulating the synchronization of the state of its instances *tbag* in several 'processes'.

In [5]:
{
    const int NUM_PROC = 5;
    const int SIZE = 10;

    templet::write_ahead_log wal;

    //////////////// 'process' simulation ///////////////////////////////
    std::atomic_int PID = 0;
    std::vector<std::thread> threads(NUM_PROC);
	for (auto& t : threads)t = std::thread([&] { int pid = PID++;
	//////////////// inside a 'process' /////////////////////////////////
    bag_of_tasks tbag(wal);

    if(pid==0){// in master 'process'
        tbag.resize(SIZE);
        for(int i=0;i<SIZE;i++) tbag.add(i,i);
    }
    while(!tbag.ready_to_get())/*wait*/;

    unsigned id; int N, NxN; 
    while(tbag.get(id,N)){
        NxN = N*N; tbag.put(id,NxN);
    }

    if(pid==0){// in master 'process'
        for(int i=0;i<SIZE;i++)
            std::cout << tbag.N[i] <<"^2 = " << tbag.NxN[i] << std::endl;
    }                                             
    ////////////// outside a 'process' //////////////////////////////////
	}); for (auto& t : threads) t.join();
	std::cout << "Success!" << std::endl;
    /////////////////////////////////////////////////////////////////////
}

0^2 = 0
1^2 = 1
2^2 = 4
3^2 = 9
4^2 = 16
5^2 = 25
6^2 = 36
7^2 = 49
8^2 = 64
9^2 = 81
Success!
